# Module 12: Reading your real code — embed & search

Module 11 gave you the *idea* of an embedding. Now you read the two real files you and Claude built — the ones that actually run on your 7,400 messages. Nothing new to install; you wrote these. The goal is to look at each line and **recognise the move**.

> **Kernel:** pick **Python (memory-venv)** in the top-right kernel picker — this module imports `chromadb`/`anthropic`, which live only in that environment.

## The learning loop (7 steps)

1. **Read** the code  2. **Predict** what it does  3. **Run** it  4. **Reflect**
5. **Ask** Claude if unclear  6. **Write** a small variant  7. **Fix** a broken version

# Lesson 1: `embed_messages.py` — filling the meaning-store

**Run** the cell to print your own file, then read it top to bottom. **Predict first:** it touches *two* separate stores — which two, and what goes in each?

In [ ]:
print(open("/Users/jenniferfletcher/Desktop/embed_messages.py").read())

**Reflect on the key moves:**
- `sqlite3.connect(...)` — opens **memory.db** (the words).
- `get_or_create_collection(..., metadata={"hnsw:space": "cosine"})` — opens **chroma_store** (the meanings), set to the cosine distance you used by hand in Module 11.
- `already = set(collection.get(...)["ids"])` — the **dedup**: which messages are already embedded, so a re-run only does the new ones.
- the `SELECT ... JOIN` — pulls each message plus its conversation name.
- `collection.add(...)` — embeds a batch and stores the vectors.

So: read words out of memory.db → embed → store vectors in chroma_store.

# Lesson 2: see it live (read-only)

This opens your **real** store and counts what's in it. **Predict:** roughly how many messages are embedded? (Reading is safe — this only counts, it doesn't change anything.)

In [ ]:
import chromadb
from chromadb.utils import embedding_functions

embed = embedding_functions.DefaultEmbeddingFunction()
collection = chromadb.PersistentClient(path="/Users/jenniferfletcher/Desktop/chroma_store").get_collection(
    "messages", embedding_function=embed
)
print("messages embedded in your memory:", collection.count())

# Lesson 3: `semantic_search.py` — asking the store a question

**Run** to print it, then read. **Predict:** where does the *cosine similarity* from Module 11 actually happen in here? (It's not a line you'll see — that's the clue.)

In [ ]:
print(open("/Users/jenniferfletcher/Desktop/semantic_search.py").read())

**Reflect:** the cosine comparison happens *inside* `collection.query(...)` — ChromaDB does it for you, against every stored vector at once. The line `similarity = 1 - dist` just turns ChromaDB's *distance* back into the *similarity* score you computed by hand in Module 11. You embed the query, Chroma finds the nearest stored vectors, you show them.

## ✍️ Write (step 6)

From the blank cell, **search your own memory**: call `collection.query(query_texts=["...your question..."], n_results=3)` and print the `conversation` name from each result's metadata. (Hint: results come back as `res["metadatas"][0]` — a list of dicts.) Predict whether your query will find anything before you run it.

In [ ]:
# your code here


---

### Now compare to a reference implementation

Below is one way to do the Write task. **Predict** what it does, run it, then compare to YOUR version. Where they differ — is one better, or are they just different styles?

In [ ]:
# A reference: ask your real memory store a real question
res = collection.query(
    query_texts=["what is my research thesis actually about"],
    n_results=3,
)

# results come back as res["metadatas"][0] -- a list of dicts (one per hit)
for meta in res["metadatas"][0]:
    print(meta["conversation"])


**📝 Reflect** — how does the reference compare to yours? Pay attention to the `[0]` after `metadatas` — that's the unwrap from "list of queries" (which you only gave one) to "list of hits." Forgetting that `[0]` is the bug in the Fix cell below — same shape, different consequence.

## 🔧 Fix (step 7)

This is **broken on purpose** — it forgets that ChromaDB wraps results in an outer list (one slot per query), so it crashes or prints nonsense. Read it, predict the error, run it, then fix it to print the three conversation names. (Hint: look for a missing `[0]`.)

In [ ]:
res = collection.query(query_texts=["how does trust develop"], n_results=3)
for meta in res["metadatas"]:        # <-- this is the list-of-lists, not the inner list
    print(meta["conversation"])

## Next

You now understand the **IN** (`embed_messages.py`) and the **OUT** (`semantic_search.py`). **Module 13** shows how these get triggered *automatically* — the hooks.